Import packages + spyglass + custom tables

In [ ]:
import datajoint as dj
dj.config.load("dj_local_conf.json")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from spyglass.lfp.analysis.v1.lfp_band import LFPBandV1

from spyglass.position.position_merge import PositionOutput
import spyglass.lfp as lfp
from spyglass.lfp.analysis.v1 import lfp_band

import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src"))
from sleep.updown_tables_dmr import UpDownStateParams, UpDownStateSelection, UpDownStates
from sleep.sleep_table_dmr import SleepScoringParams, SleepScoringSelection, SleepScoring

SleepScoring has a parameter table. We can look at the existing parameter options, and also add parameters we would like to use for sleep scoring using the ```insert``` function. Importantly, sleep_scoring_params_name must be unique.

In [ ]:
SleepScoringParams()

In [ ]:
key = {
            "sleep_scoring_params_name": "hierarchical",
            "method": "hierarchical",
            "use_hierarchical": True,
            "use_pss": False,            "power_smoothing": 0.5,
            "speed_smoothing": 0.5,
            "apply_constraints": True,
            "rem_cannot_follow_wake": True,
            "constraint_max_iterations": 15,
            "min_duration": 5.0,
            "speed_threshold": 3.0,
            "use_speed_for_wake": True,
        }

In [ ]:
SleepScoringParams().insert1(key, skip_duplicates = True)

We then have a selection table, where we can select the data we want to sleep score.

In [ ]:
SleepScoringSelection()

First, we get the data we are interested in

In [ ]:
lfp.LFPOutput.LFPV1() & {'nwb_file_name': 'Charlie20260223_.nwb'}

In [ ]:
nwb_file_name = 'Charlie20260223_.nwb' #this is Charlie's first day on the w-track with ontime opto stim
lfp_electrode_group_name = 'left and right mPFC' #since we plan on using this for more cortical analyses
interval_list_name = '05_s3' #this is the second sleep session with opto and third overall

lfp_filter = "LFP 0-400 Hz DMR"
delta_filter = 'Delta 0.5-4 Hz DMR'
theta_filter = 'Theta 5-11 Hz DMR'

sampling_rate = 1034

pos_merge_id = (PositionOutput.DLCPosV1() & {'nwb_file_name': nwb_file_name,
                             'epoch': 5}).fetch('merge_id')[0] #epoch has to match interval list name

lfp_s_key = {
    "nwb_file_name": nwb_file_name,
    "lfp_electrode_group_name": lfp_electrode_group_name,
    "target_interval_list_name": interval_list_name,
    "filter_name": lfp_filter,
    "target_sampling_rate": sampling_rate
}

lfp_merge_id = (lfp.LFPOutput.LFPV1() & lfp_s_key).fetch1("merge_id")


In [ ]:
# Again, we can insert table entries with the insert function.

key = [{'sleep_scoring_params_name': 'hierarchical',
       'lfp_merge_id': lfp_merge_id,
       'nwb_file_name': nwb_file_name,
        'filter_sampling_rate': sampling_rate,
        'target_interval_list_name': interval_list_name,
        'pos_merge_id': pos_merge_id,
        'theta_filter_name': theta_filter,
        'delta_filter_name': delta_filter,
}]
SleepScoringSelection().insert(key, skip_duplicates = True)

Now we can look to verify our entry

In [ ]:
SleepScoringSelection()

In [ ]:
SleepScoringSelection() & key

In [ ]:
#if the entry is incorrect, you can uncomment the line below and it will delete the
    #selection entry and any SleepScoring entry

# (SleepScoringSelection() & key).delete()

After verifying our entry, we can now run sleep scoring

In [ ]:
SleepScoring().populate(key)

We can verify our table entry

In [ ]:
SleepScoring() & key

We can restrict to just the session of interest with a few identifiers and then use the table results to analyze data (e.g. visualizing the sleep scoring results, getting the nrem times for downstream analysis, etc.)

In [ ]:
(SleepScoring() & {'nwb_file_name': "Charlie20260223_.nwb",
                  'target_interval_list_name': '05_s3',
                  'lfp_merge_id': lfp_merge_id}).plot_hypnogram()

In [ ]:
(SleepScoring() & {'nwb_file_name': "Charlie20260223_.nwb",
                  'target_interval_list_name': '05_s3',
                  'lfp_merge_id': lfp_merge_id}).fetch_nrem_times()